# Day 25 — Detailed Error / Failure Analysis

This notebook inspects the frozen Day 22 pH-only Linear Regression model on the same held-out chronological test partition used in Day 23. It does not retrain or tune the final model. The constant-mean baseline is included for context.

**Important:** the source dataset is static and timestamps are synthetic. This is pseudo-temporal error analysis, not validation on real underwater sensor streams.

In [ ]:
import pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA='water_potability_temporal_synthetic.csv'
MODEL='Day22_Final_Linear_Regression_Pipeline.joblib'
BASELINE='Day23_Constant_Mean_Baseline.joblib'

In [ ]:
df=pd.read_csv(DATA)
df['timestamp']=pd.to_datetime(df['timestamp'],errors='coerce')
data=df.dropna(subset=['timestamp']).sort_values('timestamp',kind='stable').reset_index(drop=True)
data['target_next_ph']=data['ph'].shift(-1)
data=data.dropna(subset=['target_next_ph']).reset_index(drop=True)
n=len(data); train_end=int(n*.70); val_end=int(n*.85)
train,val,test=data.iloc[:train_end],data.iloc[train_end:val_end],data.iloc[val_end:]
print('Train / validation / test:',len(train),len(val),len(test))

In [ ]:
model=joblib.load(MODEL)
baseline=joblib.load(BASELINE)
X=test[['ph']]
y=test['target_next_ph'].to_numpy()
pred=model.predict(X)
base_pred=baseline.predict(X)
errors=pd.DataFrame({'timestamp':test.timestamp.astype(str).to_numpy(),
 'current_ph':test.ph.to_numpy(),'actual_next_ph':y,
 'predicted_next_ph':pred,'residual_actual_minus_pred':y-pred,
 'absolute_error':np.abs(y-pred),'constant_mean_prediction':base_pred,
 'constant_mean_absolute_error':np.abs(y-base_pred)})
errors['error_direction']=np.where(errors.residual_actual_minus_pred>0,'underprediction',
 np.where(errors.residual_actual_minus_pred<0,'overprediction','exact'))
errors.sort_values('absolute_error',ascending=False).head(20)

In [ ]:
print('MAE:',mean_absolute_error(y,pred))
print('RMSE:',np.sqrt(mean_squared_error(y,pred)))
print('Mean residual:',np.mean(y-pred))
print('Maximum absolute error:',np.max(np.abs(y-pred)))
print('Rows where Linear Regression beats baseline:',
      int((errors.absolute_error<errors.constant_mean_absolute_error).sum()),'/',len(errors))

In [ ]:
# Inspect worst predictions individually
worst=errors.sort_values('absolute_error',ascending=False).head(20)
display(worst)

In [ ]:
# Group errors by actual next-row pH range (descriptive bins only)
errors['actual_ph_range']=pd.cut(errors.actual_next_ph,
 bins=[-np.inf,6.5,7.0,7.5,8.0,np.inf],
 labels=['<6.5','6.5–7.0','7.0–7.5','7.5–8.0','>8.0'])
grouped=errors.groupby('actual_ph_range',observed=False).agg(
 rows=('absolute_error','size'),
 MAE=('absolute_error','mean'),
 RMSE=('residual_actual_minus_pred',lambda x:np.sqrt(np.mean(x**2))),
 mean_residual=('residual_actual_minus_pred','mean'),
 max_error=('absolute_error','max'))
display(grouped)

In [ ]:
# Visual checks
plt.figure(figsize=(7,5))
plt.scatter(errors.actual_next_ph,errors.predicted_next_ph,alpha=.55)
lo=min(errors.actual_next_ph.min(),errors.predicted_next_ph.min())
hi=max(errors.actual_next_ph.max(),errors.predicted_next_ph.max())
plt.plot([lo,hi],[lo,hi],'--')
plt.xlabel('Actual next-row pH'); plt.ylabel('Predicted next-row pH')
plt.title('Actual vs predicted'); plt.show()

plt.figure(figsize=(7,5))
plt.hist(errors.residual_actual_minus_pred,bins=25)
plt.xlabel('Residual (actual - predicted)'); plt.ylabel('Count')
plt.title('Residual distribution'); plt.show()

plt.figure(figsize=(7,5))
plt.scatter(errors.predicted_next_ph,errors.residual_actual_minus_pred,alpha=.55)
plt.axhline(0,linestyle='--')
plt.xlabel('Predicted next-row pH'); plt.ylabel('Residual')
plt.title('Residuals vs predicted'); plt.show()

In [ ]:
# One-feature linear model explanation
linear=model.named_steps['model']
print('Feature: current-row pH')
print('Coefficient:',linear.coef_[0])
print('Intercept:',linear.intercept_)
print('Median imputation value:',model.named_steps['imputer'].statistics_[0])
print('Only one feature is used, so a multi-feature importance ranking is not applicable.')

## Limitations to carry into the final report
- Static source dataset with synthetically generated timestamps.
- Next-row pH is a pseudo-temporal target, not a genuine future sensor observation.
- Single-feature model cannot capture interactions among water-quality variables.
- Missing current pH values are median-imputed; missingness itself is not modelled.
- Abrupt changes and out-of-range inputs may cause large errors.
- No dissolved oxygen or pressure columns are present.
- No labelled anomaly ground truth; large residuals cannot be called confirmed anomalies.
- Grouped pH bins are descriptive, not regulatory/safety thresholds.
- Error patterns do not establish physical causes without verified sensor metadata.